In [ ]:
def mean_waveform(cluster_id, results_dir, n_spikes=np.inf, bfile=None, best=True):
    """Get mean waveform for `n_spikes` random spikes assigned to `cluster_id`.

    Parameters
    ----------
    cluster_id : int
        Cluster index to reference from `spike_clusters.npy` in the results
        directory. Only waveforms from spikes assigned to this cluster will
        be used.
    results_dir : str or Path
        Path to directory where Kilosort4 sorting results were saved.
    n_spikes : int; default=np.inf
        Number of spikes to use for computing mean. By default, all spikes
        assigned to `cluster_id` are used.
    bfile : kilosort.io.BinaryFiltered; optional
        Kilosort4 data file object. By default, this will be loaded using the
        information in `ops.npy` in the saved results.
    best : bool; default=True
        If True, return the mean single-channel waveform using the best channel
        for `cluster_id`. Otherwise, return the multi-channel waveform with
        all data channels included.

    Returns
    -------
    mean_wave : np.ndarray
        Mean waveform with shape (nt,) if `best = True`, or shape (n_chans,nt)
        otherwise.
    spike_subset : np.ndarray
        Indices for randomly selected spikes, relative to the list of spike
        times assigned to this cluster. In other words, `spike_subset = [0,1,2]`
        would indicate that the first, second and third spikes from this cluster
        were used (in ascending order of spike time).
    
    """
    results_dir = Path(results_dir)
    if best:
        chan = get_best_channels(results_dir)[cluster_id]
    else:
        chan = None

    spike_times, spike_subset = get_cluster_spikes(cluster_id, results_dir,
                                                   n_spikes=n_spikes)
    waves = get_spike_waveforms(spike_times, results_dir, bfile=bfile, chan=chan)
    mean_wave = waves.mean(axis=-1)

    return mean_wave, spike_subset